In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from faker import Faker
import random
import os
import pathlib

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

# Detecta raiz do projeto automaticamente
BASE_DIR = pathlib.Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

DB_PATH = BASE_DIR / 'data' / 'banco.db'

os.makedirs(BASE_DIR / 'data' / 'raw', exist_ok=True)
os.makedirs(BASE_DIR / 'data' / 'processed', exist_ok=True)
os.makedirs(BASE_DIR / 'models', exist_ok=True)
os.makedirs(BASE_DIR / 'docs', exist_ok=True)

engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

print(f"Projeto: {BASE_DIR}")
print(f"Banco:   {DB_PATH}")
print("Conexão SQLite estabelecida!")

Projeto: c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO
Banco:   c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\data\banco.db
Conexão SQLite estabelecida!


In [2]:
PROFISSOES = [
    # Gestão
    {"profissao": "Diretor Executivo (CEO)",          "setor": "Gestão",      "salario_medio": 25000},
    {"profissao": "Diretor Financeiro (CFO)",          "setor": "Gestão",      "salario_medio": 22000},
    {"profissao": "Gerente de TI",                     "setor": "Gestão",      "salario_medio": 15000},
    {"profissao": "Gerente de Banco",                  "setor": "Gestão",      "salario_medio": 12000},
    # Jurídico
    {"profissao": "Juiz",                              "setor": "Jurídico",    "salario_medio": 19000},
    {"profissao": "Promotor",                          "setor": "Jurídico",    "salario_medio": 18000},
    {"profissao": "Advogado",                          "setor": "Jurídico",    "salario_medio": 7200},
    {"profissao": "Assistente Jurídico",               "setor": "Jurídico",    "salario_medio": 3000},
    # Saúde
    {"profissao": "Médico Especialista",               "setor": "Saúde",       "salario_medio": 20000},
    {"profissao": "Médico Clínico Geral",              "setor": "Saúde",       "salario_medio": 12000},
    {"profissao": "Dentista",                          "setor": "Saúde",       "salario_medio": 6000},
    {"profissao": "Enfermeiro",                        "setor": "Saúde",       "salario_medio": 3500},
    {"profissao": "Técnico de Enfermagem",             "setor": "Saúde",       "salario_medio": 2200},
    {"profissao": "Farmacêutico",                      "setor": "Saúde",       "salario_medio": 4000},
    # Tecnologia
    {"profissao": "Engenheiro de Software",            "setor": "Tecnologia",  "salario_medio": 9000},
    {"profissao": "Cientista de Dados",                "setor": "Tecnologia",  "salario_medio": 8500},
    {"profissao": "Engenheiro de Dados",               "setor": "Tecnologia",  "salario_medio": 9500},
    {"profissao": "Analista de Sistemas",              "setor": "Tecnologia",  "salario_medio": 7000},
    {"profissao": "Analista de BI",                    "setor": "Tecnologia",  "salario_medio": 6500},
    {"profissao": "DevOps Engineer",                   "setor": "Tecnologia",  "salario_medio": 10000},
    {"profissao": "Administrador de Banco de Dados",   "setor": "Tecnologia",  "salario_medio": 8000},
    {"profissao": "Suporte Técnico",                   "setor": "Tecnologia",  "salario_medio": 3000},
    # Engenharia
    {"profissao": "Engenheiro Civil",                  "setor": "Engenharia",  "salario_medio": 8000},
    {"profissao": "Engenheiro Mecânico",               "setor": "Engenharia",  "salario_medio": 8500},
    {"profissao": "Engenheiro Elétrico",               "setor": "Engenharia",  "salario_medio": 9000},
    {"profissao": "Técnico em Edificações",            "setor": "Engenharia",  "salario_medio": 3500},
    # Financeiro
    {"profissao": "Contador",                          "setor": "Financeiro",  "salario_medio": 5000},
    {"profissao": "Analista Financeiro",               "setor": "Financeiro",  "salario_medio": 5500},
    {"profissao": "Assistente Administrativo",         "setor": "Financeiro",  "salario_medio": 2500},
    {"profissao": "Auxiliar Administrativo",           "setor": "Financeiro",  "salario_medio": 1800},
    # Comercial
    {"profissao": "Gerente Comercial",                 "setor": "Comercial",   "salario_medio": 9000},
    {"profissao": "Executivo de Vendas",               "setor": "Comercial",   "salario_medio": 6000},
    {"profissao": "Vendedor",                          "setor": "Comercial",   "salario_medio": 2000},
    {"profissao": "Representante Comercial",           "setor": "Comercial",   "salario_medio": 4000},
    # Educação
    {"profissao": "Professor Universitário",           "setor": "Educação",    "salario_medio": 8000},
    {"profissao": "Professor Ensino Médio",            "setor": "Educação",    "salario_medio": 3000},
    {"profissao": "Professor Ensino Fundamental",      "setor": "Educação",    "salario_medio": 2500},
    # Logística
    {"profissao": "Motorista",                         "setor": "Logística",   "salario_medio": 2200},
    {"profissao": "Motorista de Caminhão",             "setor": "Logística",   "salario_medio": 4000},
    {"profissao": "Operador de Empilhadeira",          "setor": "Logística",   "salario_medio": 2500},
    {"profissao": "Estoquista",                        "setor": "Logística",   "salario_medio": 1800},
    # Indústria
    {"profissao": "Operador de Máquina",               "setor": "Indústria",   "salario_medio": 2200},
    {"profissao": "Soldador",                          "setor": "Indústria",   "salario_medio": 3000},
    {"profissao": "Eletricista",                       "setor": "Indústria",   "salario_medio": 2800},
    {"profissao": "Mecânico",                          "setor": "Indústria",   "salario_medio": 3000},
    # Serviços
    {"profissao": "Atendente de Loja",                 "setor": "Serviços",    "salario_medio": 1800},
    {"profissao": "Garçom",                            "setor": "Serviços",    "salario_medio": 1700},
    {"profissao": "Cozinheiro",                        "setor": "Serviços",    "salario_medio": 2000},
    {"profissao": "Segurança",                         "setor": "Serviços",    "salario_medio": 2000},
    {"profissao": "Auxiliar de Serviços Gerais",       "setor": "Serviços",    "salario_medio": 1600},
    # Comunicação
    {"profissao": "Jornalista",                        "setor": "Comunicação", "salario_medio": 3500},
    {"profissao": "Designer Gráfico",                  "setor": "Comunicação", "salario_medio": 4000},
    {"profissao": "Publicitário",                      "setor": "Comunicação", "salario_medio": 5000},
    # Alta Renda
    {"profissao": "CEO (Grande Empresa)",              "setor": "Alta Renda",  "salario_medio": 80000},
    {"profissao": "Diretor Executivo (Multinacional)", "setor": "Alta Renda",  "salario_medio": 70000},
    {"profissao": "Diretor de Investimentos",          "setor": "Alta Renda",  "salario_medio": 65000},
    {"profissao": "Sócio Banco de Investimento",       "setor": "Alta Renda",  "salario_medio": 100000},
    {"profissao": "Trader Alta Performance",           "setor": "Alta Renda",  "salario_medio": 60000},
    {"profissao": "Hedge Fund Manager",                "setor": "Alta Renda",  "salario_medio": 120000},
    {"profissao": "Médico Cirurgião",                  "setor": "Alta Renda",  "salario_medio": 60000},
    {"profissao": "Médico Anestesista",                "setor": "Alta Renda",  "salario_medio": 50000},
    {"profissao": "Juiz Federal (Topo)",               "setor": "Alta Renda",  "salario_medio": 55000},
    {"profissao": "Promotor (Topo)",                   "setor": "Alta Renda",  "salario_medio": 50000},
    {"profissao": "Procurador da República",           "setor": "Alta Renda",  "salario_medio": 55000},
    {"profissao": "Advogado Sócio",                    "setor": "Alta Renda",  "salario_medio": 70000},
    {"profissao": "Engenheiro Software Staff (Ext.)",  "setor": "Alta Renda",  "salario_medio": 60000},
    {"profissao": "Arquiteto de Soluções Cloud",       "setor": "Alta Renda",  "salario_medio": 55000},
    {"profissao": "Especialista IA Sênior",            "setor": "Alta Renda",  "salario_medio": 65000},
    {"profissao": "Piloto Internacional",              "setor": "Alta Renda",  "salario_medio": 50000},
]

df_prof = pd.DataFrame(PROFISSOES)
df_prof['ano_referencia'] = 2024
df_prof['fonte'] = 'PNAD Contínua / Estimativa de Mercado 2024'

# Excel estilizado
wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Salários por Profissão'

ws.merge_cells('A1:E1')
ws['A1'].value = 'Tabela de Referência — Salários por Profissão (PNAD Contínua 2024)'
ws['A1'].font = Font(bold=True, size=13, color='FFFFFF')
ws['A1'].fill = PatternFill(start_color='1a1a2e', end_color='1a1a2e', fill_type='solid')
ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 30

headers = ['Profissão', 'Setor', 'Salário Médio (R$)', 'Ano Referência', 'Fonte']
for col_idx, h in enumerate(headers, 1):
    cell = ws.cell(row=2, column=col_idx, value=h)
    cell.font = Font(bold=True, color='FFFFFF', size=10)
    cell.fill = PatternFill(start_color='EC0000', end_color='EC0000', fill_type='solid')
    cell.alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[2].height = 28

for i, row in df_prof.iterrows():
    linha = i + 3
    for col_idx, val in enumerate([row['profissao'], row['setor'],
                                   row['salario_medio'], row['ano_referencia'], row['fonte']], 1):
        cell = ws.cell(row=linha, column=col_idx, value=val)
        cell.alignment = Alignment(horizontal='left', vertical='center')
        if i % 2 == 1:
            cell.fill = PatternFill(start_color='F5F5F5', end_color='F5F5F5', fill_type='solid')
    ws.row_dimensions[linha].height = 20

for col, w in zip(['A','B','C','D','E'], [40, 16, 20, 16, 40]):
    ws.column_dimensions[col].width = w

caminho_excel = str(BASE_DIR / 'data' / 'raw' / 'ibge_salarios_profissoes.xlsx')
wb.save(caminho_excel)
print(f"Excel salvo: {caminho_excel}")
print(f"Total: {len(df_prof)} profissões em {df_prof['setor'].nunique()} setores")

Excel salvo: c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\data\raw\ibge_salarios_profissoes.xlsx
Total: 69 profissões em 13 setores


In [3]:
df_ibge = pd.read_excel(caminho_excel, skiprows=1)
df_ibge.columns = ['profissao', 'setor', 'salario_medio', 'ano_referencia', 'fonte']

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_ibge_salarios"))
    conn.commit()

df_ibge.to_sql('tb_ibge_salarios', engine, if_exists='replace', index=False)

with engine.connect() as conn:
    print("=== tb_ibge_salarios ===")
    for row in conn.execute(text("""
        SELECT setor, COUNT(*) as qtd, AVG(salario_medio) as media
        FROM tb_ibge_salarios GROUP BY setor ORDER BY media DESC
    """)):
        print(f"  {row[0]:20} | {int(row[1])} profissões | média R$ {row[2]:,.0f}")

print("\nTabela criada com sucesso!")

=== tb_ibge_salarios ===
  Alta Renda           | 16 profissões | média R$ 66,562
  Gestão               | 4 profissões | média R$ 18,500
  Jurídico             | 4 profissões | média R$ 11,800
  Saúde                | 6 profissões | média R$ 7,950
  Tecnologia           | 8 profissões | média R$ 7,688
  Engenharia           | 4 profissões | média R$ 7,250
  Comercial            | 4 profissões | média R$ 5,250
  Educação             | 3 profissões | média R$ 4,500
  Comunicação          | 3 profissões | média R$ 4,167
  Financeiro           | 4 profissões | média R$ 3,700
  Indústria            | 4 profissões | média R$ 2,750
  Logística            | 4 profissões | média R$ 2,625
  Serviços             | 5 profissões | média R$ 1,820

Tabela criada com sucesso!


In [4]:
ESTADOS_CIDADES = [
    # Norte
    {"uf":"AC","estado":"Acre",               "regiao":"Norte",       "cidade":"Rio Branco",              "peso":1},
    {"uf":"AC","estado":"Acre",               "regiao":"Norte",       "cidade":"Cruzeiro do Sul",          "peso":1},
    {"uf":"AM","estado":"Amazonas",           "regiao":"Norte",       "cidade":"Manaus",                   "peso":3},
    {"uf":"AM","estado":"Amazonas",           "regiao":"Norte",       "cidade":"Parintins",                "peso":1},
    {"uf":"AP","estado":"Amapá",              "regiao":"Norte",       "cidade":"Macapá",                   "peso":1},
    {"uf":"AP","estado":"Amapá",              "regiao":"Norte",       "cidade":"Santana",                  "peso":1},
    {"uf":"PA","estado":"Pará",               "regiao":"Norte",       "cidade":"Belém",                    "peso":3},
    {"uf":"PA","estado":"Pará",               "regiao":"Norte",       "cidade":"Ananindeua",               "peso":1},
    {"uf":"PA","estado":"Pará",               "regiao":"Norte",       "cidade":"Santarém",                 "peso":1},
    {"uf":"RO","estado":"Rondônia",           "regiao":"Norte",       "cidade":"Porto Velho",              "peso":1},
    {"uf":"RR","estado":"Roraima",            "regiao":"Norte",       "cidade":"Boa Vista",                "peso":1},
    {"uf":"TO","estado":"Tocantins",          "regiao":"Norte",       "cidade":"Palmas",                   "peso":1},
    {"uf":"TO","estado":"Tocantins",          "regiao":"Norte",       "cidade":"Araguaína",                "peso":1},
    # Nordeste
    {"uf":"AL","estado":"Alagoas",            "regiao":"Nordeste",    "cidade":"Maceió",                   "peso":2},
    {"uf":"BA","estado":"Bahia",              "regiao":"Nordeste",    "cidade":"Salvador",                 "peso":8},
    {"uf":"BA","estado":"Bahia",              "regiao":"Nordeste",    "cidade":"Feira de Santana",         "peso":4},
    {"uf":"BA","estado":"Bahia",              "regiao":"Nordeste",    "cidade":"Vitória da Conquista",     "peso":2},
    {"uf":"BA","estado":"Bahia",              "regiao":"Nordeste",    "cidade":"Camaçari",                 "peso":2},
    {"uf":"CE","estado":"Ceará",              "regiao":"Nordeste",    "cidade":"Fortaleza",                "peso":4},
    {"uf":"CE","estado":"Ceará",              "regiao":"Nordeste",    "cidade":"Caucaia",                  "peso":1},
    {"uf":"MA","estado":"Maranhão",           "regiao":"Nordeste",    "cidade":"São Luís",                 "peso":2},
    {"uf":"MA","estado":"Maranhão",           "regiao":"Nordeste",    "cidade":"Imperatriz",               "peso":1},
    {"uf":"PB","estado":"Paraíba",            "regiao":"Nordeste",    "cidade":"João Pessoa",              "peso":2},
    {"uf":"PB","estado":"Paraíba",            "regiao":"Nordeste",    "cidade":"Campina Grande",           "peso":1},
    {"uf":"PE","estado":"Pernambuco",         "regiao":"Nordeste",    "cidade":"Recife",                   "peso":4},
    {"uf":"PE","estado":"Pernambuco",         "regiao":"Nordeste",    "cidade":"Caruaru",                  "peso":1},
    {"uf":"PI","estado":"Piauí",              "regiao":"Nordeste",    "cidade":"Teresina",                 "peso":2},
    {"uf":"RN","estado":"Rio Grande do Norte","regiao":"Nordeste",    "cidade":"Natal",                    "peso":2},
    {"uf":"SE","estado":"Sergipe",            "regiao":"Nordeste",    "cidade":"Aracaju",                  "peso":2},
    # Centro-Oeste
    {"uf":"DF","estado":"Distrito Federal",   "regiao":"Centro-Oeste","cidade":"Brasília",                 "peso":5},
    {"uf":"DF","estado":"Distrito Federal",   "regiao":"Centro-Oeste","cidade":"Ceilândia",                "peso":2},
    {"uf":"DF","estado":"Distrito Federal",   "regiao":"Centro-Oeste","cidade":"Taguatinga",               "peso":2},
    {"uf":"GO","estado":"Goiás",              "regiao":"Centro-Oeste","cidade":"Goiânia",                  "peso":3},
    {"uf":"GO","estado":"Goiás",              "regiao":"Centro-Oeste","cidade":"Aparecida de Goiânia",     "peso":1},
    {"uf":"MS","estado":"Mato Grosso do Sul", "regiao":"Centro-Oeste","cidade":"Campo Grande",             "peso":2},
    {"uf":"MT","estado":"Mato Grosso",        "regiao":"Centro-Oeste","cidade":"Cuiabá",                   "peso":2},
    {"uf":"MT","estado":"Mato Grosso",        "regiao":"Centro-Oeste","cidade":"Rondonópolis",             "peso":1},
    # Sudeste
    {"uf":"ES","estado":"Espírito Santo",     "regiao":"Sudeste",     "cidade":"Vitória",                  "peso":2},
    {"uf":"ES","estado":"Espírito Santo",     "regiao":"Sudeste",     "cidade":"Vila Velha",               "peso":2},
    {"uf":"MG","estado":"Minas Gerais",       "regiao":"Sudeste",     "cidade":"Belo Horizonte",           "peso":6},
    {"uf":"MG","estado":"Minas Gerais",       "regiao":"Sudeste",     "cidade":"Uberlândia",               "peso":2},
    {"uf":"MG","estado":"Minas Gerais",       "regiao":"Sudeste",     "cidade":"Contagem",                 "peso":2},
    {"uf":"MG","estado":"Minas Gerais",       "regiao":"Sudeste",     "cidade":"Juiz de Fora",             "peso":1},
    {"uf":"RJ","estado":"Rio de Janeiro",     "regiao":"Sudeste",     "cidade":"Rio de Janeiro",           "peso":12},
    {"uf":"RJ","estado":"Rio de Janeiro",     "regiao":"Sudeste",     "cidade":"Niterói",                  "peso":4},
    {"uf":"RJ","estado":"Rio de Janeiro",     "regiao":"Sudeste",     "cidade":"Petrópolis",               "peso":2},
    {"uf":"RJ","estado":"Rio de Janeiro",     "regiao":"Sudeste",     "cidade":"Volta Redonda",            "peso":2},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"São Paulo",                "peso":20},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"Campinas",                 "peso":6},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"Santos",                   "peso":4},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"Ribeirão Preto",           "peso":4},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"Sorocaba",                 "peso":3},
    {"uf":"SP","estado":"São Paulo",          "regiao":"Sudeste",     "cidade":"São José dos Campos",      "peso":3},
    # Sul
    {"uf":"PR","estado":"Paraná",             "regiao":"Sul",         "cidade":"Curitiba",                 "peso":5},
    {"uf":"PR","estado":"Paraná",             "regiao":"Sul",         "cidade":"Londrina",                 "peso":2},
    {"uf":"PR","estado":"Paraná",             "regiao":"Sul",         "cidade":"Maringá",                  "peso":2},
    {"uf":"RS","estado":"Rio Grande do Sul",  "regiao":"Sul",         "cidade":"Porto Alegre",             "peso":5},
    {"uf":"RS","estado":"Rio Grande do Sul",  "regiao":"Sul",         "cidade":"Caxias do Sul",            "peso":2},
    {"uf":"RS","estado":"Rio Grande do Sul",  "regiao":"Sul",         "cidade":"Pelotas",                  "peso":1},
    {"uf":"SC","estado":"Santa Catarina",     "regiao":"Sul",         "cidade":"Florianópolis",            "peso":8},
    {"uf":"SC","estado":"Santa Catarina",     "regiao":"Sul",         "cidade":"Joinville",                "peso":5},
    {"uf":"SC","estado":"Santa Catarina",     "regiao":"Sul",         "cidade":"Blumenau",                 "peso":4},
    {"uf":"SC","estado":"Santa Catarina",     "regiao":"Sul",         "cidade":"Chapecó",                  "peso":2},
    {"uf":"SC","estado":"Santa Catarina",     "regiao":"Sul",         "cidade":"Itajaí",                   "peso":2},
]

df_geo = pd.DataFrame(ESTADOS_CIDADES)

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_estados_cidades"))
    conn.commit()

df_geo.to_sql('tb_estados_cidades', engine, if_exists='replace', index=False)

with engine.connect() as conn:
    print("=== tb_estados_cidades ===")
    for row in conn.execute(text("""
        SELECT regiao, COUNT(DISTINCT uf) as estados, COUNT(*) as cidades, SUM(peso) as peso_total
        FROM tb_estados_cidades GROUP BY regiao ORDER BY peso_total DESC
    """)):
        print(f"  {row[0]:15} | {row[1]} estados | {row[2]} cidades | peso {row[3]}")

print(f"\nTop 5 cidades por peso:")
for _, row in df_geo.nlargest(5, 'peso').iterrows():
    print(f"  {row['cidade']:25} ({row['uf']}) | peso {row['peso']}")

=== tb_estados_cidades ===
  Sudeste         | 4 estados | 16 cidades | peso 75
  Nordeste        | 9 estados | 16 cidades | peso 40
  Sul             | 3 estados | 11 cidades | peso 38
  Centro-Oeste    | 4 estados | 8 cidades | peso 18
  Norte           | 7 estados | 13 cidades | peso 17

Top 5 cidades por peso:
  São Paulo                 (SP) | peso 20
  Rio de Janeiro            (RJ) | peso 12
  Salvador                  (BA) | peso 8
  Florianópolis             (SC) | peso 8
  Belo Horizonte            (MG) | peso 6


In [5]:
PERFIS = {
    'primeiros_passos': {
        'peso': 0.05,
        'renda_min': 800,    'renda_max': 2500,
        'idade_min': 18,     'idade_max': 26,
        'setores': ['Serviços','Logística','Indústria'],
        'desc': 'Jovem iniciando carreira, renda baixa, poucos produtos'
    },
    'trajetoria_crescente': {
        'peso': 0.35,
        'renda_min': 2500,   'renda_max': 8000,
        'idade_min': 25,     'idade_max': 45,
        'setores': ['Financeiro','Educação','Comercial','Engenharia'],
        'desc': 'No mercado há anos, relacionamento longo, produtos clássicos'
    },
    'potencial_oculto': {
        'peso': 0.30,
        'renda_min': 6000,   'renda_max': 20000,
        'idade_min': 28,     'idade_max': 50,
        'setores': ['Tecnologia','Saúde','Comunicação'],
        'desc': 'Ganha bem mas pouco explorado em produtos bancários'
    },
    'self_made': {
        'peso': 0.25,
        'renda_min': 15000,  'renda_max': 60000,
        'idade_min': 30,     'idade_max': 55,
        'setores': ['Gestão','Jurídico','Tecnologia'],
        'desc': 'Alta renda, tem produtos mas precisa de mais'
    },
    'old_money': {
        'peso': 0.05,
        'renda_min': 50000,  'renda_max': 120000,
        'idade_min': 40,     'idade_max': 70,
        'setores': ['Alta Renda'],
        'desc': 'Consolidado, carteira completa, foco em upgrades'
    },
}

NOMES_CLUSTERS = {
    'primeiros_passos':    'Primeiros Passos',
    'trajetoria_crescente':'Trajetória Crescente',
    'potencial_oculto':    'Potencial Oculto',
    'self_made':           'Self Made',
    'old_money':           'Old Money',
}

print("Perfis configurados:")
for nome, cfg in PERFIS.items():
    print(f"  {NOMES_CLUSTERS[nome]:22} | {cfg['peso']*100:.0f}% | R$ {cfg['renda_min']:,} — R$ {cfg['renda_max']:,} | {cfg['idade_min']}-{cfg['idade_max']} anos")

Perfis configurados:
  Primeiros Passos       | 5% | R$ 800 — R$ 2,500 | 18-26 anos
  Trajetória Crescente   | 35% | R$ 2,500 — R$ 8,000 | 25-45 anos
  Potencial Oculto       | 30% | R$ 6,000 — R$ 20,000 | 28-50 anos
  Self Made              | 25% | R$ 15,000 — R$ 60,000 | 30-55 anos
  Old Money              | 5% | R$ 50,000 — R$ 120,000 | 40-70 anos


In [6]:
def sortear_profissao(perfil_cfg, df_prof):
    df_f = df_prof[df_prof['setor'].isin(perfil_cfg['setores'])]
    if len(df_f) == 0:
        df_f = df_prof
    return df_f.sample(1).iloc[0]

def sortear_localidade(df_geo):
    row = df_geo.sample(1, weights='peso').iloc[0]
    return row['uf'], row['estado'], row['regiao'], row['cidade']

def gerar_score(renda, inadimplente, idade):
    base = 400 + (renda / 120000) * 450 + (idade / 70) * 100
    if inadimplente:
        base -= np.random.uniform(100, 250)
    return int(np.clip(base + np.random.normal(0, 30), 0, 1000))

def gerar_produtos(perfil_nome, renda, score):
    probs = {
        'primeiros_passos':    {'tem_cartao_credito':0.35,'tem_credito_pessoal':0.15,'tem_financiamento':0.05,'tem_investimento':0.08,'tem_seguro':0.05,'tem_consorcio':0.03,'tem_previdencia':0.03,'tem_credito_imobiliario':0.00,'tem_cambio':0.00,'tem_cheque_especial':0.20},
        'trajetoria_crescente':{'tem_cartao_credito':0.75,'tem_credito_pessoal':0.40,'tem_financiamento':0.35,'tem_investimento':0.30,'tem_seguro':0.45,'tem_consorcio':0.20,'tem_previdencia':0.25,'tem_credito_imobiliario':0.10,'tem_cambio':0.05,'tem_cheque_especial':0.35},
        'potencial_oculto':    {'tem_cartao_credito':0.85,'tem_credito_pessoal':0.30,'tem_financiamento':0.30,'tem_investimento':0.40,'tem_seguro':0.30,'tem_consorcio':0.15,'tem_previdencia':0.25,'tem_credito_imobiliario':0.20,'tem_cambio':0.15,'tem_cheque_especial':0.20},
        'self_made':           {'tem_cartao_credito':0.95,'tem_credito_pessoal':0.25,'tem_financiamento':0.45,'tem_investimento':0.75,'tem_seguro':0.55,'tem_consorcio':0.25,'tem_previdencia':0.60,'tem_credito_imobiliario':0.45,'tem_cambio':0.40,'tem_cheque_especial':0.10},
        'old_money':           {'tem_cartao_credito':0.99,'tem_credito_pessoal':0.10,'tem_financiamento':0.20,'tem_investimento':0.95,'tem_seguro':0.80,'tem_consorcio':0.30,'tem_previdencia':0.90,'tem_credito_imobiliario':0.70,'tem_cambio':0.80,'tem_cheque_especial':0.00},
    }
    p = {'tem_conta_corrente': 1}
    for col, prob in probs[perfil_nome].items():
        p[col] = int(random.random() < prob)
    p['qtd_produtos'] = sum(v for k,v in p.items() if k.startswith('tem_'))
    p['upgrade_cartao_black']        = int(random.random() < (0.80 if p['tem_cartao_credito'] and renda>15000 else 0.0))
    p['upgrade_investimento_plus']   = int(random.random() < (0.70 if p['tem_investimento'] and perfil_nome in ['self_made','old_money'] else 0.0))
    p['upgrade_limite_credito']      = int(random.random() < (0.50 if p['tem_credito_pessoal'] and score>600 else 0.0))
    p['upgrade_credito_imobiliario'] = int(random.random() < (0.40 if p['tem_financiamento'] and renda>6000 else 0.0))
    p['upgrade_limite_cheque']       = int(random.random() < (0.45 if p['tem_cheque_especial'] and score>500 else 0.0))
    p['upgrade_conta_global']        = int(random.random() < (0.60 if p['tem_cambio'] and perfil_nome in ['self_made','old_money'] else 0.0))
    p['qtd_upgrades']                = sum(v for k,v in p.items() if k.startswith('upgrade_'))
    return p

def gerar_comportamento(renda, perfil_nome):
    txn = int(renda / 300)
    inadim_prob = {'primeiros_passos':0.15,'trajetoria_crescente':0.08,'potencial_oculto':0.05,'self_made':0.03,'old_money':0.01}
    meses_rel   = {'primeiros_passos':(1,12),'trajetoria_crescente':(24,180),'potencial_oculto':(12,120),'self_made':(36,240),'old_money':(60,240)}
    mn, mx = meses_rel[perfil_nome]
    return {
        'media_transacoes_mes':   max(1, int(np.random.normal(txn, txn*0.3))),
        'saldo_medio':            round(np.random.lognormal(np.log(max(100, renda*0.4)), 0.5), 2),
        'uso_app_dias_mes':       min(30, max(0, int(np.random.normal(22 if perfil_nome=='primeiros_passos' else 14, 5)))),
        'meses_relacionamento':   random.randint(mn, mx),
        'teve_inadimplencia_12m': int(random.random() < inadim_prob[perfil_nome]),
        'volume_cambio_usd':      round(np.random.lognormal(6, 0.8), 2) if perfil_nome in ['self_made','old_money'] else 0.0,
    }

print("Funções definidas!")

Funções definidas!


In [7]:
df_prof_db = pd.read_sql('SELECT * FROM tb_ibge_salarios', engine)
df_geo_db  = pd.read_sql('SELECT * FROM tb_estados_cidades', engine)

perfil_nomes = list(PERFIS.keys())
pesos        = [PERFIS[p]['peso'] for p in perfil_nomes]
registros    = []

print("Gerando 300.000 clientes...")
for i in range(300000):
    if i % 50000 == 0 and i > 0:
        print(f"  {i:,} gerados...")

    perfil_nome = random.choices(perfil_nomes, weights=pesos, k=1)[0]
    cfg         = PERFIS[perfil_nome]
    prof        = sortear_profissao(cfg, df_prof_db)
    uf, estado, regiao, cidade = sortear_localidade(df_geo_db)

    renda_base = float(prof['salario_medio'])
    renda      = round(float(np.clip(
        np.random.lognormal(np.log(max(800, renda_base)), 0.25),
        cfg['renda_min'], cfg['renda_max']
    )), 2)

    idade  = random.randint(cfg['idade_min'], cfg['idade_max'])
    inadim = random.random() < 0.08
    score  = gerar_score(renda, inadim, idade)
    prods  = gerar_produtos(perfil_nome, renda, score)
    comp   = gerar_comportamento(renda, perfil_nome)

    registros.append({
        'id_cliente':         f'CLI{str(i+1).zfill(6)}',
        'nome':               fake.name(),
        'email':              fake.email(),
        'cpf':                fake.cpf(),
        'data_nascimento':    fake.date_of_birth(minimum_age=idade, maximum_age=idade).strftime('%Y-%m-%d'),
        'idade':              idade,
        'telefone':           fake.phone_number(),
        'uf':                 uf,
        'estado':             estado,
        'regiao':             regiao,
        'cidade':             cidade,
        'profissao':          prof['profissao'],
        'setor':              prof['setor'],
        'salario_referencia': float(prof['salario_medio']),
        'renda_mensal':       renda,
        'score_credito':      score,
        'perfil_real':        perfil_nome,
        **prods,
        **comp,
    })

df_clientes = pd.DataFrame(registros)
print(f"\nGerados: {len(df_clientes):,} clientes | {len(df_clientes.columns)} colunas")
print(f"Missing values: {df_clientes.isnull().sum().sum()}")
print(f"\nDistribuição de perfis:")
print(df_clientes['perfil_real'].value_counts())

Gerando 300.000 clientes...


  50,000 gerados...


  100,000 gerados...


  150,000 gerados...


  200,000 gerados...


  250,000 gerados...



Gerados: 300,000 clientes | 42 colunas
Missing values: 0

Distribuição de perfis:
perfil_real
trajetoria_crescente    105295
potencial_oculto         89578
self_made                74821
old_money                15255
primeiros_passos         15051
Name: count, dtype: int64


In [8]:
print("Salvando no SQLite...")

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS tb_clientes"))
    conn.commit()

df_clientes.to_sql('tb_clientes', engine, if_exists='replace', index=False, chunksize=10000)

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM tb_clientes")).scalar()
    print(f"Total: {total:,} clientes")

    print("\nDistribuição de perfis:")
    for row in conn.execute(text("""
        SELECT perfil_real, COUNT(*) as qtd, ROUND(AVG(renda_mensal),0) as renda_media
        FROM tb_clientes GROUP BY perfil_real ORDER BY renda_media DESC
    """)):
        print(f"  {row[0]:25} | {row[1]:,} clientes | R$ {row[2]:,.0f}")

    print("\nDistribuição por região:")
    for row in conn.execute(text("""
        SELECT regiao, COUNT(*) as qtd FROM tb_clientes
        GROUP BY regiao ORDER BY qtd DESC
    """)):
        print(f"  {row[0]:15} | {row[1]:,}")

    print("\nDistribuição de setores:")
    for row in conn.execute(text("""
        SELECT setor, COUNT(*) as qtd, ROUND(AVG(renda_mensal),0) as media
        FROM tb_clientes GROUP BY setor ORDER BY media DESC
    """)):
        print(f"  {row[0]:20} | {row[1]:,} | R$ {row[2]:,.0f}")

print(f"\nBanco: {DB_PATH}")

Salvando no SQLite...


Total: 300,000 clientes

Distribuição de perfis:
  old_money                 | 15,255 clientes | R$ 69,543
  self_made                 | 74,821 clientes | R$ 16,874
  potencial_oculto          | 89,578 clientes | R$ 8,279
  trajetoria_crescente      | 105,295 clientes | R$ 5,061
  primeiros_passos          | 15,051 clientes | R$ 2,100

Distribuição por região:


  Sudeste         | 119,747
  Nordeste        | 63,865
  Sul             | 60,590
  Centro-Oeste    | 28,718
  Norte           | 27,080

Distribuição de setores:
  Alta Renda           | 15,255 | R$ 69,543
  Gestão               | 18,713 | R$ 20,180
  Jurídico             | 18,703 | R$ 17,263
  Tecnologia           | 79,608 | R$ 11,508
  Saúde                | 31,518 | R$ 9,233
  Engenharia           | 28,157 | R$ 6,532
  Comunicação          | 15,857 | R$ 6,088
  Comercial            | 27,858 | R$ 5,107
  Educação             | 21,078 | R$ 4,425
  Financeiro           | 28,202 | R$ 4,021
  Indústria            | 4,729 | R$ 2,343
  Logística            | 4,608 | R$ 2,181
  Serviços             | 5,714 | R$ 1,834

Banco: c:\Users\caleb\Documents\SANTANDER-MOTOR-RECOMENDACAO\data\banco.db


In [9]:
env_path = BASE_DIR / '.env'

if not env_path.exists():
    with open(env_path, 'w', encoding='utf-8') as f:
        f.write("# Configurações do projeto — preencha antes de rodar o notebook 08\n\n")
        f.write("# Banco de dados\n")
        f.write("DATABASE_URL=sqlite:///data/banco.db\n\n")
        f.write("# Email (Mailtrap)\n")
        f.write("MAILTRAP_HOST=sandbox.smtp.mailtrap.io\n")
        f.write("MAILTRAP_PORT=587\n")
        f.write("MAILTRAP_USER=\n")
        f.write("MAILTRAP_PASS=\n\n")
        f.write("# Destinatário do relatório\n")
        f.write("EMAIL_REMETENTE=\n")
        f.write("EMAIL_DESTINATARIO=\n")
    print(f".env criado em: {env_path}")
    print("Abra e preencha as credenciais antes de rodar o notebook 08!")
else:
    print(".env já existe!")

.env já existe!
